## Ray Serve Observability Guide

This guide helps you diagnose and tune Ray Serve applications in production using the Serve Grafana Dashboard. Rather than overwhelming you with all available metrics, it presents goal-oriented flows through the dashboard.

**Dashboard Sections:**
1. Serve Overview - Cluster health at a glance
2. Deployment Deep Dive - Per-deployment request flow analysis
3. Autoscaling and Capacity - Scaling behavior and infrastructure health
4. Request Batching - Batch processing efficiency
5. Model Multiplexing - Multi-model serving performance

## Flow 1: "Is My Application Healthy?"

Start here when you need a quick health check or first notice something might be wrong.

### Step 1: Check the Serve Overview Section

Open the **Serve Overview** row (expanded by default) and scan these key indicators:

| Panel | Healthy State | Action If Not |
|-------|---------------|---------------|
| **Application Status Timeline** | Shows `6` (RUNNING) | If `1` (DEPLOY_FAILED) or `2` (UNHEALTHY), check controller logs |
| **Error Rate %** | Green (< 1%) | If yellow/red (> 1-5%), investigate errors in Deployment Deep Dive |
| **P99 HTTP/gRPC Latency** | Within your SLA | If high, drill into latency breakdown |
| **Healthy Proxies** | Matches expected count | If fewer, check proxy status in Autoscaling section |

See screenshot below:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_overview_tab.png" width="1000">

### Step 2: Check Cluster Resources (Hardware Utilization)

Scan the cluster resource gauges at the bottom:

- **Cluster CPU/GPU/Memory/Disk** - If any gauge is > 70% (yellow) or > 90% (red), your cluster may need more capacity
- **Head Node gauges** - Head node resource exhaustion can impact the Serve controller

See screenshot below:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_cluster_resources_tab.png" width="1000">


## Suggested Decision Tree

Here's a suggested decision tree for diagnosing issues:

- Error Rate high?  Go to Flow 2 (**Debugging Errors**)
- Latency high but Error Rate low?  Go to Flow 3 (**Latency Analysis**)
- Autoscaling not working?  Go to Flow 4 (**Autoscaling Debug**)
- Everything looks fine?  Your application is healthy

For optimizations like **dynamic batching** and **model multiplexing**, see Flows 5 and 6.

## Flow 2: "Why Are Requests Failing?"

Use this flow when you see elevated error rates.

### Step 1: Identify Error Source

Open **Deployment Deep Dive** and check:

1. **Error Rate % (stat panel)** - Confirms error percentage at replica level
2. **Error QPS per Deployment** - Shows which deployment(s) have errors

### Step 2: Determine Error Type

Ray Serve tracks errors at different layers. Compare these metrics:

| Metric | What It Measures | Common Causes |
|--------|------------------|---------------|
| **Error QPS per Deployment** | Exceptions in your code | User code bugs, dependency failures |
| **Error QPS per Application** in Overview | HTTP-level errors at proxy | Timeouts, routing failures, backpressure |

**If errors are at the replica level** (Error QPS per Deployment is high):
- Check replica logs for exception tracebacks

**If errors are at the proxy level** (Error QPS per Application is high but Error QPS per Deployment is low):
- Likely timeout or routing issue
- Check **Queued Requests at Router** - if high, requests are timing out waiting for replicas
- Check **Scheduling Tasks in Backoff** - if high, no replicas are available


### Step 3: Check for Backpressure

In Deployment Deep Dive:
- **Queued Requests at Router** - Requests waiting to be assigned
- **Assigned Requests (Router View)** - Requests sent to replicas
- **Processing Requests (Replica View)** - Requests actually being processed

**High queued requests indicate:**
1. Not enough replicas for the load
2. `max_ongoing_requests` is too low (replicas reject new requests)
3. Individual requests are taking too long

See screenshot below for a closer look at the router view:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_deployment_deep_dive_router_view.png" width="1000">

## Flow 3: "Why Is Latency High?"

Use this flow to identify latency bottlenecks.

### Understanding Latency Layers

Ray Serve measures latency at multiple points. Understanding the difference is critical:

**With HTTP Proxy (default):**

1. **Proxy** receives request → measures **P50/P90/P99 E2E Latency** (total time from receipt to response)
2. **Router** queues request → measures **P50/P90/P99 Router Fulfillment Time** (time waiting for replica assignment)
3. **Replica** processes request → measures **P50/P90/P99 Processing Latency** (your code execution time)

**With HAProxy (direct routing):**

1. **HAProxy** routes directly to replica (no queue metrics at this layer)
2. **Replica** processes request → measures **P50/P90/P99 Processing Latency**

*Note: If using deployment composition (one deployment calls another), the downstream deployment still goes through a Router with queue metrics.*

### Step 1: Compare E2E vs Processing Latency

In **Serve Overview**:
- Check **P50/P90/P99 E2E Latency** panels

See screenshot below:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_e2e_latency_tab.png" width="1000">

In **Deployment Deep Dive**:
- Check **P50/P90/P99 Processing Latency per Deployment** panels

**Calculate the gap:**
```
Queue Wait Time ≈ E2E Latency - Processing Latency
```

**If Processing Latency is high (close to E2E):**
- Your code is slow
- Consider optimizing your model/business logic
- Check if batching could help (Flow 5)

**If Queue Wait Time is high (E2E >> Processing):**
- Requests are waiting too long for a replica
- Check **Router Fulfillment Time** panels (P50/P90/P99)
- This confirms requests are queuing

See screenshot below:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_processing_and_fulfillment_latency_tab.png" width="1000">

### Step 2: Diagnose Queue Wait Time

In **Deployment Deep Dive**, check:

1. **Queued Requests at Router** - If growing, demand exceeds capacity
2. **Scheduling Tasks in Backoff** - If > 0, scheduler can't find available replicas

**Root causes and fixes:**

| Symptom | Root Cause | Fix |
|---------|------------|-----|
| Queued requests growing, all replicas at `max_ongoing_requests` | Insufficient replicas | Increase `max_replicas` or lower `target_ongoing_requests` |
| Queued requests growing, replicas have capacity | Routing inefficiency | Check **Load Balance Quality** panel |
| Scheduling tasks in backoff high | No replicas available | Check replica health in Autoscaling section |


### Step 3: Check Load Balance Quality

The **Load Balance Quality (Max/Avg)** panel shows how traffic is distributed across replicas:
- Close to 1.0 = even distribution
- Much > 1.0 = some replicas receive more traffic than others

**Unbalanced load is expected and desirable when using:**
- **Model multiplexing** - Requests route to replicas with the model already loaded (avoids cache misses)
- **Locality-aware routing** - Requests prefer same-node/AZ replicas (reduces network latency)
- **Sticky sessions** - Applications requiring session affinity

**Unbalanced load may indicate a problem when:**
- You're NOT using any of the above features, but some replicas are consistently hotter
- Hot replicas are becoming overloaded while others are idle

## Flow 4: "Why Isn't Autoscaling Working?"

Use this flow when you observe these symptoms in earlier sections:
- **Deployment Deep Dive**: **Queued Requests at Router** growing steadily
- **Deployment Deep Dive**: **Deployment Status Timeline** stuck at UPSCALING (4) for an extended period
- **Serve Overview**: High latency or growing **Ongoing HTTP/gRPC Requests** despite expecting autoscaling to help

### Step 1: Check Autoscaling Decision vs Reality

Under the **Autoscaling** section, check:

1. **Target vs Actual Replicas Over Time**
   - **Target** line = what the autoscaler decided
   - **Actual** line = healthy replicas running
   - Gap = "autoscaling lag" (time to provision replicas)

2. **Autoscaling Decision vs Target**
   - **Desired** = raw autoscaling decision
   - **Target** = after applying min/max bounds
   - If they differ, you're hitting `min_replicas` or `max_replicas` limits

See screenshot below:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_autoscaling_tab.png" width="1000">

### Step 2: Understand Autoscaling Inputs

Check **Total Requests (Autoscaler View)**:
- This is what the autoscaler sees: queued + in-flight requests
- The autoscaling formula is: `desired_replicas = total_requests / target_ongoing_requests`

**If total requests is low but you expect more:**
- Metrics might be delayed - check **Metrics Delay** stat
- Check **Replica Metrics Delay** and **Handle Metrics Delay** graphs

### Step 3: Check Metrics Pipeline Health

**Metrics Delay** panel shows max delay for autoscaling metrics. If high:
1. Check **Replica Metrics Delay** - delay from replicas to controller
2. Check **Handle Metrics Delay** - delay from handles to controller
3. High delays mean the autoscaler is working with stale data

**Long Poll Latency** affects how quickly routers learn about new replicas:
- Check **Long Poll Latency** (P50/P99)
- High latency delays autoscaling responsiveness

### Step 4: Check Why New Replicas Are Slow to Start

If there's a gap between Target and Actual replicas:

1. **Replica Startup (P99)** stat - Total time to start a replica
2. **Replica Startup Latency** graph - Breakdown over time
3. **Replica Initialization Latency** - Time in your `__init__` method

**Common slow startup causes:**
- Large model loading in `__init__`
- Runtime environment setup (pip install, Docker pull)
- Node provisioning for cluster autoscaling

### Step 5: Check Controller Health

The controller runs the autoscaling control loop:

1. **Control Loop Duration** - Time per iteration (should be < 1s)
2. **Control Loop Rate** - Iterations per second

**If control loop is slow:**
- Controller may be overloaded
- Check head node resource utilization

## Flow 5: "Is My Batching Efficient?"

Use this flow when using the `@serve.batch` decorator.

### Step 1: Check Batch Utilization

In **Request Batching** section:

1. **Batch Utilization %** stat - Are batches filling up?
   - Close to 100% = optimal
   - Low (< 50%) = batches timing out before filling

2. **Median Batch Size** - Actual batch sizes

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_batching_tab.png" width="1000">

### Step 2: Diagnose Low Utilization

Check **Batch Wait Time** panels:
- P50/P99 wait time tells you how long requests wait for a batch

**If batch utilization is low and wait time equals `batch_wait_timeout_s`:**
- Traffic is too sparse to fill batches
- Consider: lower `max_batch_size`, shorter `batch_wait_timeout_s`, or fewer replicas

**If batch utilization is low but wait time is short:**
- Batches are dispatching early
- Check if a custom `batch_size_fn` is limiting batch sizes

### Step 3: Check Batch Throughput

1. **Batches Processed per Second** - How many batches execute
2. **Batch Queue Length** - Requests waiting in batch queue
   - If growing, batching can't keep up
   - Consider increasing `max_concurrent_batches`

3. **Batch Execution Time** - Time to execute your batch function
   - If high, optimize your batch processing code

## Flow 6: "Is My Model Multiplexing Efficient?"

Use this flow when using `@serve.multiplexed`.

### Step 1: Check Cache Efficiency

In **Model Multiplexing** section:

1. **Model Cache Hit Rate** - Most critical metric
   - > 80% = healthy (green)
   - 50-80% = some thrashing (yellow)
   - < 50% = poor cache efficiency (red)

2. **Models Loaded (Cluster-Wide)** - Total models in memory

See screenshot below:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_multiplexing_tab.png" width="1000">

### Step 2: Diagnose Cache Misses

If cache hit rate is low:

1. Check **Model Load/Unload Rate** - High churn indicates:
   - Working set larger than `max_num_models_per_replica`
   - Traffic pattern has poor locality

2. Check **Model Evictions** graph - Rate of evictions over time

3. Check **Loaded Models per Deployment** - Are replicas at capacity?

### Step 3: Understand Loading Performance

1. **P99 Model Load Time** - Time to load models
   - If high, consider model lazy-loading or smaller models

2. **P99 Model Unload Latency** - Time to unload models
   - If high, check cleanup logic

**Tuning guidance:**
- If cache hit rate is low but you have memory: increase `max_num_models_per_replica`
- If load latency is high: consider pre-warming models or using faster storage

## Flow 7: "Debugging Event Loop Issues"

Use this flow when you suspect:

1. async code is blocking.
2. control loop is overloaded.
3. system is slow.

### When to Check

Check system metrics when you see:
- High latency despite low replica processing time
- Requests timing out without apparent cause
- Uneven load distribution

### Step 1: Check System Health

Under the **System Metrics** section, for example to check event loop health, see:

1. **Event Loop Scheduling Latency (P99)** stat:
   - < 10ms = healthy (green)
   - 10-50ms = acceptable under load (yellow)
   - 50-100ms = concerning (orange)
   - > 100ms = problematic (red)

2. **Event Loop Tasks** - Pending asyncio tasks
   - High/growing = task accumulation

See screenshot below:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-serve-deep-dive/serve_system_metrics_tab.png" width="1000">

### Step 2: Identify Blocking Code

If event loop latency is high:

1. **Event Loop Scheduling Latency by Component** - Shows which component is blocked:
   - `replica` component = your deployment code
   - `proxy` component = proxy handlers

2. **Event Loop Monitoring Heartbeat** - Iterations per second
   - Drop to zero = event loop is completely blocked

**Common causes:**
- Synchronous I/O in async handlers
- CPU-bound code in async context
- Blocking third-party library calls

**Fix:** Move blocking operations to a thread pool using `asyncio.to_thread()` or `loop.run_in_executor()`.